In [3]:
r"Database=PortfolioProject_MarketingAnalytics;"

'Database=PortfolioProject_MarketingAnalytics;'

In [7]:
# Required libraries:
# pandas, nltk, pyodbc, sqlalchemy

import pandas as pd
import pyodbc
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer


# Download VADER sentiment dictionary
nltk.download('vader_lexicon')


# Connect to SQL Server and load customer reviews
def fetch_data_from_sql():

    conn_str = (
        r"Driver={SQL Server};"
        r"Server=RAKSHANA\SQLEXPRESS;"
        r"Database=PortfolioProject_MarketingAnalytics;"
        r"Trusted_Connection=yes;"
    )

    conn = pyodbc.connect(conn_str)

    query = """
    SELECT
        ReviewID,
        CustomerID,
        ProductID,
        ReviewDate,
        Rating,
        ReviewText
    FROM dbo.customer_reviews
    """

    df = pd.read_sql(query, conn)

    conn.close()

    return df


# Load review data
customer_reviews_df = fetch_data_from_sql()


# Initialize sentiment analyzer
sia = SentimentIntensityAnalyzer()


# Calculate the compound sentiment score for each review
def calculate_sentiment(review):

    if pd.isna(review):
        return 0

    review = str(review)

    sentiment = sia.polarity_scores(review)

    return sentiment['compound']


# Classify reviews using sentiment score and rating
def categorize_sentiment(score, rating):

    if score > 0.05:

        if rating >= 4:
            return 'Positive'
        elif rating == 3:
            return 'Mixed Positive'
        else:
            return 'Mixed Negative'

    elif score < -0.05:

        if rating <= 2:
            return 'Negative'
        elif rating == 3:
            return 'Mixed Negative'
        else:
            return 'Mixed Positive'

    else:

        if rating >= 4:
            return 'Positive'
        elif rating <= 2:
            return 'Negative'
        else:
            return 'Neutral'


# Group sentiment scores into ranges
def sentiment_bucket(score):

    if score >= 0.5:
        return '0.5 to 1.0'
    elif 0.0 <= score < 0.5:
        return '0.0 to 0.49'
    elif -0.5 <= score < 0.0:
        return '-0.49 to 0.0'
    else:
        return '-1.0 to -0.5'


# Generate sentiment scores
customer_reviews_df['SentimentScore'] = (
    customer_reviews_df['ReviewText']
    .apply(calculate_sentiment)
)


# Assign sentiment categories
customer_reviews_df['SentimentCategory'] = (
    customer_reviews_df.apply(
        lambda row: categorize_sentiment(
            row['SentimentScore'],
            row['Rating']
        ),
        axis=1
    )
)


# Create sentiment score ranges
customer_reviews_df['SentimentBucket'] = (
    customer_reviews_df['SentimentScore']
    .apply(sentiment_bucket)
)


# Preview the processed data
print("\nCustomer Reviews with Sentiment Analysis:")
print(customer_reviews_df.head())


# Review sentiment distribution
print("\nSentiment Category Counts:")
print(customer_reviews_df['SentimentCategory'].value_counts())


# Export the final dataset
customer_reviews_df.to_csv(
    'customer_reviews_with_sentiment.csv',
    index=False
)

print("\nSentiment analysis completed successfully!")
print("File saved as: customer_reviews_with_sentiment.csv")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\raksh\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
C:\Users\raksh\AppData\Local\Temp\ipykernel_19876\1287818805.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)



Customer Reviews with Sentiment Analysis:
   ReviewID  CustomerID  ProductID  ReviewDate  Rating  \
0         1          77         18  2023-12-23       3   
1         2          80         19  2024-12-25       5   
2         3          50         13  2025-01-26       4   
3         4          78         15  2025-04-21       3   
4         5          64          2  2023-07-16       3   

                                 ReviewText  SentimentScore SentimentCategory  \
0   Average  experience,  nothing  special.         -0.3089    Mixed Negative   
1            The  quality  is    top-notch.          0.0000          Positive   
2   Five  stars  for  the  quick  delivery.          0.0000          Positive   
3  Good  quality,  but  could  be  cheaper.          0.2382    Mixed Positive   
4   Average  experience,  nothing  special.         -0.3089    Mixed Negative   

  SentimentBucket  
0    -0.49 to 0.0  
1     0.0 to 0.49  
2     0.0 to 0.49  
3     0.0 to 0.49  
4    -0.49 to 0.0  

